In [21]:
import pandas as pd 
import numpy as np
from pandas_datareader import wb
from datetime import datetime
import time

In [22]:
countries = ['BR', 'RU', 'IN', 'MX','KR', 'ZA',
    'ID', 'TH', 'PH', 'HU',          
    'CL', 'CO', 'PE', 'EG']

# Dropped due to missing data: ['Czechia', 'Nigeria', 'Poland', 'Turkiye', 'Vietnam']

indicators = {
        'PA.NUS.FCRF': 'fx_rate',
        'GC.DOD.TOTL.GD.ZS': 'debt_to_gdp',
        'FP.CPI.TOTL.ZG': 'inflation',
        'NY.GDP.MKTP.KD.ZG': 'gdp_growth', 
        'BN.CAB.XOKA.GD.ZS': 'current_account_balance',
        'FR.INR.RINR': 'real_interest_rate',
    }

start_year = 2000
end_year = 2025

In [23]:
all_countries_data = []
for idx, country in enumerate(countries, 1):

    for attempt in range(3):
        try:
            country_df = wb.download(
                indicator=list(indicators.keys()),
                country=[country], 
                start=start_year,
                end=end_year
            )
            all_countries_data.append(country_df)
            break
        except Exception as e:
            if attempt < 2:
                time.sleep(3)

/var/folders/p4/7btq0jlx7m578jftvnfvgkxm0000gn/T/ipykernel_5627/622362848.py:6: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  country_df = wb.download(
/var/folders/p4/7btq0jlx7m578jftvnfvgkxm0000gn/T/ipykernel_5627/622362848.py:6: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  country_df = wb.download(
/var/folders/p4/7btq0jlx7m578jftvnfvgkxm0000gn/T/ipykernel_5627/622362848.py:6: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  country_df = wb.download(
/var/folders/p4/7btq0jlx7m578jftvnfvgkxm0000gn/T/ipykernel_5627/622362848.py:6: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `

In [24]:
raw_data = pd.concat(all_countries_data)

df = raw_data.rename(columns=indicators).reset_index()
df['year'] = df['year'].astype(int)
df = df.sort_values(['country', 'year']).set_index(['country', 'year'], drop=True)

df['debt_to_gdp_lag1'] = df.groupby(level='country')['debt_to_gdp'].shift(1)
df['inflation_lag1'] = df.groupby(level='country')['inflation'].shift(1)
df['gdp_growth_lag1'] = df.groupby(level='country')['gdp_growth'].shift(1)
df['current_account_balance_lag1'] = df.groupby(level='country')['current_account_balance'].shift(1)
df['real_interest_rate_lag1'] = df.groupby(level='country')['real_interest_rate'].shift(1)


df['log_fx'] = np.log(df['fx_rate'])
df['fx_depreciation'] = df.groupby(level='country')['log_fx'].diff()

feature_cols = ['debt_to_gdp', 'inflation', 'gdp_growth', 'current_account_balance', 'real_interest_rate']
df[feature_cols] = df.groupby(level='country')[feature_cols].ffill()

feature_cols_with_lags = ['debt_to_gdp', 'inflation', 'gdp_growth', 'current_account_balance', 'real_interest_rate', 'debt_to_gdp_lag1', 'inflation_lag1', 'gdp_growth_lag1', 'current_account_balance_lag1', 'real_interest_rate_lag1']
df[feature_cols_with_lags] = df.groupby(level='country')[feature_cols_with_lags].ffill()

feature_cols_lags = ['debt_to_gdp_lag1', 'inflation_lag1', 'gdp_growth_lag1', 'current_account_balance_lag1', 'real_interest_rate_lag1']
df[feature_cols_lags] = df.groupby(level='country')[feature_cols_lags].ffill()

In [25]:
#Econometrics Analysis Henceforth
import statsmodels.api as sm
from linearmodels.panel import PanelOLS, PooledOLS

In [26]:
x = df[feature_cols]
y = df['fx_depreciation']
x_with_const = sm.add_constant(x)
x_with_lags = df[feature_cols_with_lags]
x_lags = df[feature_cols_lags]

In [27]:
#Pooled OLS Model as a baseline
pooled_model = PooledOLS(y, x_with_const)
pooled_results = pooled_model.fit(cov_type='clustered', cluster_entity=True)
print(f'Pooled OLS Results:\n{pooled_results.summary}\n')

Pooled OLS Results:
                          PooledOLS Estimation Summary                          
Dep. Variable:        fx_depreciation   R-squared:                        0.2583
Estimator:                  PooledOLS   R-squared (Between):              0.7522
No. Observations:                 317   R-squared (Within):               0.2170
Date:                Sun, Jul 12 2026   R-squared (Overall):              0.2583
Time:                        18:29:42   Log-likelihood                    335.69
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      21.663
Entities:                          14   P-value                           0.0000
Avg Obs:                       22.643   Distribution:                   F(5,311)
Min Obs:                       14.000                                           
Max Obs:                       25.000   F-statistic (robust):             7.1075
        

/opt/anaconda3/lib/python3.13/site-packages/linearmodels/panel/model.py:919: MissingValueWarning: 
Inputs contain missing values. Dropping rows with missing observations.
  super().__init__(dependent, exog, weights=weights, check_rank=check_rank)


In [28]:
#Controlling for country fixed effects
cfe_model = PanelOLS(y, x, entity_effects=True)
cfe_results = cfe_model.fit(cov_type='clustered', cluster_entity=True)
print(f'Country Fixed Effects Results:\n{cfe_results.summary}\n')

Country Fixed Effects Results:
                          PanelOLS Estimation Summary                           
Dep. Variable:        fx_depreciation   R-squared:                        0.2282
Estimator:                   PanelOLS   R-squared (Between):              0.5354
No. Observations:                 317   R-squared (Within):               0.2282
Date:                Sun, Jul 12 2026   R-squared (Overall):              0.2726
Time:                        18:29:42   Log-likelihood                    342.38
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      17.622
Entities:                          14   P-value                           0.0000
Avg Obs:                       22.643   Distribution:                   F(5,298)
Min Obs:                       14.000                                           
Max Obs:                       25.000   F-statistic (robust):             5.13

/opt/anaconda3/lib/python3.13/site-packages/linearmodels/panel/model.py:1258: MissingValueWarning: 
Inputs contain missing values. Dropping rows with missing observations.
  super().__init__(dependent, exog, weights=weights, check_rank=check_rank)


In [29]:
#Controlling for time fixed effects
tfe_model = PanelOLS(y, x, time_effects=True)
tfe_results = tfe_model.fit(cov_type='clustered', cluster_time=True)
print(f'Time Fixed Effects Results:\n{tfe_results.summary}\n')

Time Fixed Effects Results:
                          PanelOLS Estimation Summary                           
Dep. Variable:        fx_depreciation   R-squared:                        0.2770
Estimator:                   PanelOLS   R-squared (Between):              0.8709
No. Observations:                 317   R-squared (Within):               0.1723
Date:                Sun, Jul 12 2026   R-squared (Overall):              0.2789
Time:                        18:29:42   Log-likelihood                    397.17
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      21.993
Entities:                          14   P-value                           0.0000
Avg Obs:                       22.643   Distribution:                   F(5,287)
Min Obs:                       14.000                                           
Max Obs:                       25.000   F-statistic (robust):             20.096


/opt/anaconda3/lib/python3.13/site-packages/linearmodels/panel/model.py:1258: MissingValueWarning: 
Inputs contain missing values. Dropping rows with missing observations.
  super().__init__(dependent, exog, weights=weights, check_rank=check_rank)


In [30]:
#Controlling for time and country fixed effects
tcfe_model = PanelOLS(y, x, entity_effects=True, time_effects=True)
tcfe_results = tcfe_model.fit(cov_type='clustered', cluster_entity=True, cluster_time=True)
print(f'Time and Country Fixed Effects Results:\n{tcfe_results.summary}\n')


Time and Country Fixed Effects Results:
                          PanelOLS Estimation Summary                           
Dep. Variable:        fx_depreciation   R-squared:                        0.2267
Estimator:                   PanelOLS   R-squared (Between):              0.3480
No. Observations:                 317   R-squared (Within):               0.1789
Date:                Sun, Jul 12 2026   R-squared (Overall):              0.2033
Time:                        18:29:42   Log-likelihood                    404.10
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      16.069
Entities:                          14   P-value                           0.0000
Avg Obs:                       22.643   Distribution:                   F(5,274)
Min Obs:                       14.000                                           
Max Obs:                       25.000   F-statistic (robust):        

/opt/anaconda3/lib/python3.13/site-packages/linearmodels/panel/model.py:1258: MissingValueWarning: 
Inputs contain missing values. Dropping rows with missing observations.
  super().__init__(dependent, exog, weights=weights, check_rank=check_rank)


In [31]:
# Controlling for country and time fixed effects and with lagged independent variables
ltcfe_model = PanelOLS(y, x_lags, entity_effects=True, time_effects=True)
ltcfe_results = ltcfe_model.fit(cov_type='clustered', cluster_entity=True, cluster_time=True)
print(f'Lagged Time and Country Fixed Effects Results:\n{ltcfe_results.summary}\n')

Lagged Time and Country Fixed Effects Results:
                          PanelOLS Estimation Summary                           
Dep. Variable:        fx_depreciation   R-squared:                        0.0309
Estimator:                   PanelOLS   R-squared (Between):             -1.1883
No. Observations:                 312   R-squared (Within):               0.0148
Date:                Sun, Jul 12 2026   R-squared (Overall):             -0.2108
Time:                        18:29:42   Log-likelihood                    362.38
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      1.7164
Entities:                          14   P-value                           0.1310
Avg Obs:                       22.286   Distribution:                   F(5,269)
Min Obs:                       13.000                                           
Max Obs:                       25.000   F-statistic (robust): 

/opt/anaconda3/lib/python3.13/site-packages/linearmodels/panel/model.py:1258: MissingValueWarning: 
Inputs contain missing values. Dropping rows with missing observations.
  super().__init__(dependent, exog, weights=weights, check_rank=check_rank)


In [32]:
# Testing how results change when we drop Russia from the dataset, due to its currency crisis causing potential outliers. Testing for robustness of results.
excluded_country = 'Russian Federation'
df_without_ru = df.drop(index=excluded_country, level='country', errors='ignore')

x_without_ru = df_without_ru[feature_cols_lags]
y_without_ru = df_without_ru['fx_depreciation']

tcfe_model_without_ru = PanelOLS(y_without_ru, x_without_ru, entity_effects=True, time_effects=True)
tcfe_results_without_ru = tcfe_model_without_ru.fit(cov_type='clustered', cluster_entity=True, cluster_time=True)
print(f'Time and Country Fixed Effects Results without Russia:\n{tcfe_results_without_ru.summary}\n')

Time and Country Fixed Effects Results without Russia:
                          PanelOLS Estimation Summary                           
Dep. Variable:        fx_depreciation   R-squared:                        0.0317
Estimator:                   PanelOLS   R-squared (Between):             -1.2730
No. Observations:                 288   R-squared (Within):               0.0210
Date:                Sun, Jul 12 2026   R-squared (Overall):             -0.2241
Time:                        18:29:42   Log-likelihood                    341.10
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      1.6084
Entities:                          13   P-value                           0.1584
Avg Obs:                       22.154   Distribution:                   F(5,246)
Min Obs:                       13.000                                           
Max Obs:                       25.000   F-statistic (r

/opt/anaconda3/lib/python3.13/site-packages/linearmodels/panel/model.py:1258: MissingValueWarning: 
Inputs contain missing values. Dropping rows with missing observations.
  super().__init__(dependent, exog, weights=weights, check_rank=check_rank)
